# ⚽ CapixaStat · POC de Visão Computacional (YOLOv8)

Detecção e **rastreamento** de jogadores e bola sobre o vídeo de uma partida do futebol capixaba,
com exportação de **vídeo anotado** (cada jogador com um ID).

**Módulo de Ciência de Dados — Projeto Integrador (FAESA).**

---
### ⚠️ Antes de começar
1. **Ative a GPU**: menu `Runtime` → `Change runtime type` → *Hardware accelerator* = **GPU (T4)**.
2. Rode as células **na ordem**.

### 🎥 Leitura honesta sobre o que dá pra extrair
A transmissão usa **câmera móvel** (gira e dá zoom) e o vídeo é 2D. Então **posição no campo, distância,
velocidade e mapa de calor não são confiáveis** com esse tipo de material — por isso esta POC foca em
**detecção + rastreamento** (que funcionam bem em qualquer clipe). Métricas táticas de verdade exigiriam
uma **câmera tática fixa** ou **calibração por homografia** (ver a última seção).

## 1 · Instalação

In [ ]:
!pip -q install ultralytics yt-dlp
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__, "| GPU:", torch.cuda.is_available())

## 2 · Obter um trecho do jogo

Baixa o vídeo do YouTube e corta um **trecho curto** (um lance). Ajuste `START` e `DUR`.
Se preferir, pule esta célula e use a célula de **upload manual** logo abaixo.

In [ ]:
# Link do jogo (Capixaba x Rio Branco ES) — troque se quiser outro lance/partida
YOUTUBE_URL = "https://www.youtube.com/live/0vK9-uccR6E"
START = "00:21:30"   # início do trecho (hh:mm:ss) — ajuste para o lance que quer analisar
DUR   = 20            # duração do trecho em segundos (mantenha curto: 10–25s)

# 1) baixa o jogo em até 720p
!yt-dlp -f "bestvideo[height<=720]+bestaudio/best[height<=720]" -o jogo_full.mp4 "{YOUTUBE_URL}"
# 2) corta o trecho (sem áudio, re-encodado para ficar leve)
!ffmpeg -y -ss {START} -i jogo_full.mp4 -t {DUR} -c:v libx264 -an clip.mp4
CLIP = "clip.mp4"
print("Trecho salvo em:", CLIP)

In [ ]:
# (ALTERNATIVA) Upload manual de um MP4 — descomente para usar
# from google.colab import files
# up = files.upload()
# CLIP = list(up.keys())[0]
# print("Usando:", CLIP)

## 3 · Detecção + rastreamento (YOLOv8 + ByteTrack)

Detecta **jogadores** (classe `person`) e **bola** (classe `sports ball`), atribui um **ID** a cada jogador
e gera o vídeo anotado. Guardamos as posições (pés) de cada ID para o mapa de calor.

In [ ]:
from ultralytics import YOLO
import cv2, numpy as np

# yolov8m = bom equilíbrio precisão/velocidade. Use "yolov8n.pt" (mais rápido) ou "yolov8x.pt" (mais preciso).
model = YOLO("yolov8m.pt")

# dimensões/fps do clipe
cap = cv2.VideoCapture(CLIP)
FPS = cap.get(cv2.CAP_PROP_FPS) or 25
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

writer = cv2.VideoWriter("anotado.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H))
ids_vistos = set()

# classes do COCO: 0 = person, 32 = sports ball
results = model.track(source=CLIP, stream=True, persist=True,
                      classes=[0, 32], conf=0.30, tracker="bytetrack.yaml", verbose=False)

for r in results:
    frame = r.orig_img.copy()
    b = r.boxes
    if b is not None and b.id is not None:
        ids  = b.id.cpu().numpy().astype(int)
        xyxy = b.xyxy.cpu().numpy()
        clss = b.cls.cpu().numpy().astype(int)
        for (x1, y1, x2, y2), tid, c in zip(xyxy, ids, clss):
            is_ball = (c == 32)
            color = (0, 165, 255) if is_ball else (118, 230, 0)  # BGR
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            label = "bola" if is_ball else f"#{tid}"
            cv2.putText(frame, label, (int(x1), int(y1) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            if not is_ball:
                ids_vistos.add(int(tid))
    writer.write(frame)

writer.release()
print("Vídeo anotado salvo: anotado.mp4")
print("IDs de pessoas rastreadas:", len(ids_vistos), "(inclui árbitros/reservas — o YOLO marca qualquer pessoa)")

## 4 · Ver / baixar o vídeo anotado

In [ ]:
# re-encoda para H.264 (toca no navegador) e mostra inline
!ffmpeg -y -i anotado.mp4 -vcodec libx264 -movflags +faststart anotado_web.mp4 -loglevel error
from IPython.display import HTML
from base64 import b64encode
mp4 = open("anotado_web.mp4", "rb").read()
data = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=720 controls><source src="{data}" type="video/mp4"></video>')

In [ ]:
# baixar o vídeo anotado
from google.colab import files
files.download("anotado_web.mp4")

## 5 · Separar os times (cor da camisa)

Pega **um quadro** do lance (por padrão, acha sozinho o mais cheio — tipo um escanteio) e separa os
jogadores em **dois times** medindo **quanto de cor forte** (ex.: vermelho) cada camisa tem.

> Ajustado para **um time de cor forte × um time claro** (caso deste jogo: vermelho × branco). Para outra
> combinação, ajuste a faixa de cor na função `frac_cor`. Árbitro/goleiro contam no time de cor mais parecida.
> É uma leitura de apoio, não exata.

In [ ]:
import numpy as np
from google.colab.patches import cv2_imshow

# None = acha sozinho o quadro mais cheio. Ou defina um tempo em segundos, ex.: FRAME_TIME = 30
FRAME_TIME = None

# fracao de "cor forte" (padrao = vermelho) no torso. Para azul use (H>95)&(H<135);
# verde (H>40)&(H<85); amarelo (H>18)&(H<35) — sempre com S>70 e V>40.
def frac_cor(crop):
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    Hc, Sc, Vc = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    forte = ((Hc < 12) | (Hc > 168)) & (Sc > 70) & (Vc > 40)   # vermelho
    return float(forte.mean())

cap = cv2.VideoCapture(CLIP)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
Hh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); Ww = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
if FRAME_TIME is None:
    best = (0, None, 0); i = 0; step = max(1, int(fps))
    while True:
        ok, fr = cap.read()
        if not ok: break
        if i % step == 0:
            rr = model(fr, classes=[0], conf=0.35, verbose=False)[0]
            bb = rr.boxes.xyxy.cpu().numpy() if rr.boxes is not None else np.zeros((0,4))
            big = [b for b in bb if (b[3]-b[1]) > Hh*0.11]
            if len(big) > best[0]: best = (len(big), fr.copy(), i)
        i += 1
    frame, fidx = best[1], best[2]
else:
    fidx = int(FRAME_TIME*fps); cap.set(cv2.CAP_PROP_POS_FRAMES, fidx); _, frame = cap.read()
cap.release()
print("Quadro usado: ~%.1fs" % (fidx/fps))

# detecta pessoas e mede a cor da camisa (torso)
r = model(frame, classes=[0], conf=0.35, verbose=False)[0]
players, fr_cor = [], []
for (x1,y1,x2,y2) in r.boxes.xyxy.cpu().numpy():
    w, h = x2-x1, y2-y1
    if h < Hh*0.11: continue   # ignora gente distante / arquibancada
    cx1, cx2 = int(x1+0.25*w), int(x2-0.25*w)
    cy1, cy2 = int(y1+0.12*h), int(y1+0.45*h)
    crop = frame[max(0,cy1):cy2, max(0,cx1):cx2]
    if crop.size == 0: continue
    players.append((int(x1),int(y1),int(x2),int(y2))); fr_cor.append(frac_cor(crop))

# separa em 2 times pelo quanto de cor forte cada um tem
feat = np.float32(fr_cor).reshape(-1,1)
crit = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)
_, labels, centers = cv2.kmeans(feat, 2, None, crit, 10, cv2.KMEANS_PP_CENTERS)
labels = labels.flatten()
colorido = int(np.argmax(centers.flatten()))   # cluster com mais cor forte
names = {colorido: ("Time A", (60,60,255)), 1-colorido: ("Time B", (0,230,118))}

counts = {"Time A":0, "Time B":0}; annot = frame.copy()
for (x1,y1,x2,y2), lab in zip(players, labels):
    nm, col = names[lab]; counts[nm] += 1
    cv2.rectangle(annot, (x1,y1), (x2,y2), col, 2)
    cv2.putText(annot, nm, (x1,y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
txt = "Time A (cor forte): %d   Time B (claro): %d" % (counts["Time A"], counts["Time B"])
cv2.rectangle(annot, (0,0), (Ww,40), (0,0,0), -1)
cv2.putText(annot, txt, (12,26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
cv2.imwrite("times.jpg", annot)
print(txt)
cv2_imshow(annot)

In [ ]:
# baixar o print dos times
from google.colab import files
files.download("times.jpg")

## 6 · Limitações e o que dá pra evoluir

**Por que não há mapa de calor / distância / velocidade aqui:** a transmissão usa **câmera móvel** (gira e dá zoom)
e trabalhamos em imagem 2D. Logo, posição no campo, distância e heatmap **não são confiáveis** com esse tipo de vídeo.
O YOLO também marca **qualquer pessoa** (bandeirinha, gandula, reservas), não só jogadores.

Caminhos para extrair informação de verdade:
- **Separar por time (cor da camisa)**: mostrar a forma/linha de cada equipe num quadro pausado e filtrar quem não é jogador.
- **Homografia**: calibrar 4 pontos do campo → converter a imagem em coordenadas reais (aí sim heatmap/distância fazem sentido).
- **Câmera tática fixa**: com uma câmera lateral fixa, tracking e métricas ficam diretamente aproveitáveis.
- **Auto-telestração**: usar a detecção para posicionar sozinho os marcadores no Quadro Tático do CapixaStat.